# Agentic AI Network Investigation Team

A vendor-agnostic multi-agent framework for autonomous network investigation and root cause analysis.

---

## Notebook Sections

0. Project Setup
1. Imports
2. Shared Types
3. Configuration
4. Domain Model
5. Evidence Graph
6. Evidence Adapters
7. Deterministic Executors
8. AI Agents
9. Investigation Workflow
10. Validation

DESIGN PHILOSOPHY

Agents reason.

Executors execute.

Evidence adapters translate.

The Evidence Graph is the single source of truth.

Agents never interact directly with vendor-specific data.

#0. PROJECT SETUP

Connect to github, clone repo if necessary, etc.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import pprint

from google.colab import drive


# ============================================================
# PROJECT SETTINGS
# ============================================================

REPO_NAME = "agentic-ai-network-investigation-team"
GITHUB_USER = "icarovazquez"
BRANCH = "main"

DRIVE_MOUNT = Path("/content/drive")
PROJECTS_DIR = DRIVE_MOUNT / "MyDrive" / "Colab Notebooks"
PROJECT_ROOT = PROJECTS_DIR / REPO_NAME

# Store the persistent key in Google Drive.
DRIVE_SSH_DIR = DRIVE_MOUNT / "MyDrive" / ".ssh_colab"
DRIVE_PRIVATE_KEY = DRIVE_SSH_DIR / "id_ed25519"
DRIVE_PUBLIC_KEY = DRIVE_SSH_DIR / "id_ed25519.pub"

# Runtime SSH directory. Colab resets this when the runtime restarts.
RUNTIME_SSH_DIR = Path.home() / ".ssh"
RUNTIME_PRIVATE_KEY = RUNTIME_SSH_DIR / "id_ed25519"
RUNTIME_PUBLIC_KEY = RUNTIME_SSH_DIR / "id_ed25519.pub"
SSH_CONFIG = RUNTIME_SSH_DIR / "config"
KNOWN_HOSTS = RUNTIME_SSH_DIR / "known_hosts"

SSH_REPO_URL = (
    f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
)


# ============================================================
# COMMAND HELPER
# ============================================================

def run_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess:
    """Run a shell command and display its output."""

    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"{' '.join(command)}"
        )

    return result


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("✓ Google Drive is already mounted.")


# ============================================================
# 2. CREATE A PERSISTENT SSH KEY IF NEEDED
# ============================================================

DRIVE_SSH_DIR.mkdir(parents=True, exist_ok=True)

new_key_created = False

if not DRIVE_PRIVATE_KEY.exists():
    print("Creating a persistent SSH key in Google Drive...")

    run_command(
        [
            "ssh-keygen",
            "-t",
            "ed25519",
            "-C",
            "icarovazquez-colab",
            "-f",
            str(DRIVE_PRIVATE_KEY),
            "-N",
            "",
        ]
    )

    new_key_created = True
    print("✓ Persistent SSH key created.")
else:
    print("✓ Persistent SSH key already exists in Google Drive.")


# ============================================================
# 3. COPY THE KEY INTO THE COLAB RUNTIME
# ============================================================

RUNTIME_SSH_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_PRIVATE_KEY, RUNTIME_PRIVATE_KEY)
shutil.copy2(DRIVE_PUBLIC_KEY, RUNTIME_PUBLIC_KEY)

os.chmod(RUNTIME_SSH_DIR, 0o700)
os.chmod(RUNTIME_PRIVATE_KEY, 0o600)
os.chmod(RUNTIME_PUBLIC_KEY, 0o644)

print("✓ SSH key copied into the Colab runtime.")


# ============================================================
# 4. CREATE SSH CONFIGURATION
# ============================================================

SSH_CONFIG.write_text(
    f"""Host github.com
    HostName github.com
    User git
    IdentityFile {RUNTIME_PRIVATE_KEY}
    IdentitiesOnly yes
"""
)

os.chmod(SSH_CONFIG, 0o600)

print("✓ SSH configuration created.")


# ============================================================
# 5. ADD GITHUB TO KNOWN_HOSTS
# ============================================================

keyscan = subprocess.run(
    ["ssh-keyscan", "-t", "ed25519", "github.com"],
    text=True,
    capture_output=True,
    check=True,
)

KNOWN_HOSTS.write_text(keyscan.stdout)
os.chmod(KNOWN_HOSTS, 0o644)

print("✓ GitHub added to known_hosts.")


# ============================================================
# FIRST-RUN GITHUB REGISTRATION
# ============================================================

if new_key_created:
    print("\n" + "=" * 70)
    print("ONE-TIME GITHUB SETUP REQUIRED")
    print("=" * 70)
    print(
        "Copy the public key below and add it at:\n"
        "GitHub → Settings → SSH and GPG keys → New SSH key\n"
    )

    print(DRIVE_PUBLIC_KEY.read_text().strip())

    print(
        "\nAfter adding the key to GitHub, rerun this cell."
    )

    raise SystemExit(
        "SSH key created. Add the public key to GitHub, then rerun."
    )


# ============================================================
# TEST GITHUB AUTHENTICATION
# ============================================================

ssh_test = run_command(
    [
        "ssh",
        "-o",
        "StrictHostKeyChecking=yes",
        "-T",
        "git@github.com",
    ],
    check=False,
)

# GitHub returns exit code 1 even when authentication succeeds.
ssh_output = (
    ssh_test.stdout + "\n" + ssh_test.stderr
).lower()

if "successfully authenticated" not in ssh_output:
    raise RuntimeError(
        "GitHub SSH authentication failed.\n"
        "Confirm that the displayed public key was added to your "
        "GitHub account under SSH and GPG keys."
    )

print("✓ GitHub SSH authentication succeeded.")


# ============================================================
# 6. CLONE OR UPDATE THE REPOSITORY
# ============================================================

PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    print(f"Cloning repository into:\n{PROJECT_ROOT}")

    run_command(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            SSH_REPO_URL,
            str(PROJECT_ROOT),
        ]
    )

elif not (PROJECT_ROOT / ".git").exists():
    raise RuntimeError(
        f"{PROJECT_ROOT} exists but is not a Git repository."
    )

else:
    print(f"✓ Repository already exists at:\n{PROJECT_ROOT}")

    # Ensure the remote uses SSH rather than HTTPS.
    run_command(
        [
            "git",
            "remote",
            "set-url",
            "origin",
            SSH_REPO_URL,
        ],
        cwd=PROJECT_ROOT,
    )

    status = run_command(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
    )

    if status.stdout.strip():
        print(
            "\nLocal changes detected. Git pull was skipped so that "
            "uncommitted work is not overwritten."
        )
    else:
        print(f"Pulling the latest origin/{BRANCH} changes...")

        run_command(
            [
                "git",
                "pull",
                "--ff-only",
                "origin",
                BRANCH,
            ],
            cwd=PROJECT_ROOT,
        )


# ============================================================
# ENTER THE PROJECT DIRECTORY
# ============================================================

os.chdir(PROJECT_ROOT)

print("\n" + "=" * 70)
print("PROJECT INITIALIZATION COMPLETE")
print("=" * 70)
print(f"Project root: {PROJECT_ROOT}")

run_command(["git", "remote", "-v"], cwd=PROJECT_ROOT)
run_command(["git", "status", "--short", "--branch"], cwd=PROJECT_ROOT)

Mounted at /content/drive
✓ Persistent SSH key already exists in Google Drive.
✓ SSH key copied into the Colab runtime.
✓ SSH configuration created.
✓ GitHub added to known_hosts.
Hi icarovazquez! You've successfully authenticated, but GitHub does not provide shell access.
✓ GitHub SSH authentication succeeded.
✓ Repository already exists at:
/content/drive/MyDrive/Colab Notebooks/agentic-ai-network-investigation-team
M "notebooks/Agentic AI Network Investigation Team.ipynb"

Local changes detected. Git pull was skipped so that uncommitted work is not overwritten.

PROJECT INITIALIZATION COMPLETE
Project root: /content/drive/MyDrive/Colab Notebooks/agentic-ai-network-investigation-team
origin	git@github.com:icarovazquez/agentic-ai-network-investigation-team.git (fetch)
origin	git@github.com:icarovazquez/agentic-ai-network-investigation-team.git (push)
## main...origin/main
 M "notebooks/Agentic AI Network Investigation Team.ipynb"


CompletedProcess(args=['git', 'status', '--short', '--branch'], returncode=0, stdout='## main...origin/main\n M "notebooks/Agentic AI Network Investigation Team.ipynb"\n', stderr='')

#1. IMPORTS & SHARED UTILITIES

In [2]:
!pip install --upgrade aisuite aisuite[anthropic] #Andrew Ng's wrapper lib that calls different LLM provides and abstracts each providers' API needs
!pip install anthropic #for calling anthropic LLMs through aisuite
!pip install -U "langfuse>2.0.0" #obervability & evals

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: docstring-parser
    Found existing installation: docstring_parser 0.18.0
    Uninstalling docstring_parser-0.18.0:
      Successfully uninstalled docstring_parser-0.18.0
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 2.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [45]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional
from uuid import uuid4
from abc import ABC, abstractmethod
from google.colab import userdata


import networkx as nx
import aisuite as ai
import ast

#initialize ai client
Client = ai.Client()

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

DEFAULT_AGENT_MODEL = "anthropic:claude-haiku-4-5"

AGENT_MODEL_MAP = {
    "incident_framing_agent": DEFAULT_AGENT_MODEL,
    "hypothesis_generator_agent": DEFAULT_AGENT_MODEL,
}

AGENT_MAX_TOKENS = {
    "incident_framing_agent": 2500,
    "hypothesis_generator_agent": 2000,
}

LLM_USAGE = []

#2. OBSERVABILITY & LLMs

In [5]:
from langfuse import Langfuse, get_client
from langfuse import observe

LANGFUSE_PUBLIC_KEY = userdata.get("LANGFUSE_PUBLIC_KEY_NI_TEAM").strip()
LANGFUSE_SECRET_KEY = userdata.get("LANGFUSE_SECRET_KEY_NI_TEAM").strip()
LANGFUSE_BASE_URL = "https://cloud.langfuse.com"

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    base_url=LANGFUSE_BASE_URL,
)

try:
    projects = langfuse.api.projects.get()
    print("✅ Connected. Projects:", [p.name for p in projects.data])
except Exception as e:
    print("❌ Langfuse connection failed")
    print("Type:", type(e).__name__)
    print("Message:", str(e))

✅ Connected. Projects: ['agentic-ai-network-investigation-team']


In [6]:
def trim_messages(messages, max_chars=12000):

    trimmed = []
    total_chars = 0

    # start from newest messages first
    for msg in reversed(messages):

        content = msg.get("content", "")

        if not isinstance(content, str):
            content = str(content)

        remaining = max_chars - total_chars

        if remaining <= 0:
            break

        # trim oversized message if needed
        if len(content) > remaining:
            content = content[-remaining:]

        trimmed.append({
            **msg,
            "content": content
        })

        total_chars += len(content)

    # restore chronological order
    return list(reversed(trimmed))

In [7]:
@observe(name="llm_call", as_type='generation')
def llm_call(agent_name:str, messages: list, temperature: float = 1.0, tools: list= None) -> str:

  """
  observability wrapper that all agents will use when calling the llm
  logs model, input, output, and token usage to Langfuse
  """

  selected_model = AGENT_MODEL_MAP.get(agent_name, DEFAULT_AGENT_MODEL)

  max_tokens = AGENT_MAX_TOKENS.get(agent_name, 3000)

  messages = trim_messages(messages, max_chars=12000)

  call_kwargs = {
    "model": selected_model,
    "messages": messages,
    "temperature": temperature,
    "max_tokens": max_tokens,
  }

  if tools:
    call_kwargs["tools"] = tools

  #calling the llm
  response = Client.chat.completions.create(**call_kwargs)
  content = response.choices[0].message.content

  #log output after calling the LLM
  usage = getattr(response, "usage", None)

  input_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
  output_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
  total_tokens = input_tokens + output_tokens

  result = {
        "agent_name": agent_name,
        "model": selected_model,
        "temperature": temperature,
        "content": content,
        "usage": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens
        } if usage else {},
        "tools_available": bool(tools),
    }

  LLM_USAGE.append({
    "agent": agent_name,
    "model": selected_model,
    "input_tokens": input_tokens,
    "output_tokens": output_tokens,
    "total_tokens": total_tokens})

  print('llm call completed')

  return result


In [8]:
def clean_llm_dict_output(raw_output: str) -> str:

    cleaned = raw_output.strip()

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        if lines[0].strip().startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    # Extract only the dictionary
    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start != -1 and end != -1:
        cleaned = cleaned[start:end + 1]

    return cleaned

#3. SHARED TYPES

In [9]:
from enum import Enum


class EntityType(str, Enum):
    """Types of entities that can appear in the Evidence Graph."""

    SITE = "site"
    DEVICE = "device"
    INTERFACE = "interface"
    LINK = "link"
    HOST = "host"
    SERVICE = "service"
    APPLICATION = "application"
    NETWORK = "network"
    VRF = "vrf"
    VLAN = "vlan"
    ROUTE = "route"


class RelationshipType(str, Enum):
    """Relationships between entities."""

    CONTAINS = "contains"
    CONNECTED_TO = "connected_to"
    DEPENDS_ON = "depends_on"
    ROUTES_TO = "routes_to"
    HOSTS = "hosts"
    MEMBER_OF = "member_of"
    PEERS_WITH = "peers_with"
    BACKS_UP = "backs_up"


class EvidenceType(str, Enum):
    """Evidence categories."""

    TOPOLOGY = "topology"
    TELEMETRY = "telemetry"
    CONFIGURATION = "configuration"
    EVENT = "event"
    CHANGE = "change"
    FLOW = "flow"
    SYNTHETIC_TEST = "synthetic_test"
    ROUTING = "routing"


class InvestigationStatus(str, Enum):
    """Overall investigation lifecycle."""

    CREATED = "created"
    IN_PROGRESS = "in_progress"
    WAITING_FOR_EVIDENCE = "waiting_for_evidence"
    ROOT_CAUSE_IDENTIFIED = "root_cause_identified"
    REMEDIATION_RECOMMENDED = "remediation_recommended"
    CLOSED = "closed"


class HypothesisStatus(str, Enum):
    """Status of an investigation hypothesis."""

    PROPOSED = "proposed"
    SUPPORTED = "supported"
    WEAKENED = "weakened"
    REJECTED = "rejected"
    CONFIRMED = "confirmed"


class EvidenceDirection(str, Enum):
    """How evidence affects a hypothesis."""

    SUPPORTS = "supports"
    CONTRADICTS = "contradicts"
    NEUTRAL = "neutral"


class RemediationMode(str, Enum):
    """How remediation is handled."""

    RECOMMEND_ONLY = "recommend_only"
    HUMAN_APPROVAL = "human_approval"
    AUTOMATIC = "automatic"


class SourceFormat(str, Enum):
    """Physical or logical format exposed by an evidence source."""

    CSV = "csv"
    JSON = "json"
    JSONL = "jsonl"
    PARQUET = "parquet"
    GRAPHML = "graphml"
    YAML = "yaml"
    API = "api"
    DATABASE = "database"
    COMMAND = "command"
    LIVE_LAB = "live_lab"


class AccessMode(str, Enum):
    """How an adapter accesses an evidence source."""

    FILE = "file"
    API = "api"
    DATABASE = "database"
    COMMAND = "command"
    LIVE_LAB = "live_lab"

#4. CONFIGURATION

The configuration layer describes one reproducible network investigation.

It defines:

- the incident seed;
- available evidence sources;
- source adapters and access methods;
- investigation policies;
- optional benchmark ground truth; and
- output settings.

Configuration objects describe the investigation environment. They do not load evidence or make diagnostic decisions.

##Incident Seed

In [10]:
@dataclass
class IncidentSeed:
    """
    Initial incident information supplied to the investigation.

    This is not the evolving IncidentRecord. It is the reproducible
    starting point used by the Incident Framing Agent.
    """

    reported_symptom: str
    incident_start_time: str

    affected_service: Optional[str] = None
    reported_locations: List[str] = field(default_factory=list)
    reported_entities: List[str] = field(default_factory=list)

    source: str = "benchmark"
    initial_severity: Optional[str] = None
    initial_context: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.reported_symptom.strip():
            raise ValueError("reported_symptom cannot be empty.")

        if not self.incident_start_time.strip():
            raise ValueError("incident_start_time cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

##Evidence Source Configuration

In [11]:
@dataclass
class EvidenceSourceConfig:
    """
    Configuration for one raw evidence source.

    The adapter_name identifies the translator that converts the raw
    source into vendor-neutral Evidence Graph domain records.
    """

    source_id: str
    evidence_type: EvidenceType
    source_format: SourceFormat
    access_mode: AccessMode
    adapter_name: str

    path: Optional[str] = None
    url: Optional[str] = None

    connection_parameters: Dict[str, Any] = field(
        default_factory=dict
    )
    query_parameters: Dict[str, Any] = field(
        default_factory=dict
    )

    timestamp_column: Optional[str] = None
    entity_id_columns: List[str] = field(
        default_factory=list
    )
    schema_mapping: Dict[str, str] = field(
        default_factory=dict
    )

    required: bool = True
    enabled: bool = True

    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

    def validate(self) -> None:
        if not self.source_id.strip():
            raise ValueError("source_id cannot be empty.")

        if not self.adapter_name.strip():
            raise ValueError(
                f"Source '{self.source_id}' requires adapter_name."
            )

        if (
            self.access_mode == AccessMode.FILE
            and not self.path
        ):
            raise ValueError(
                f"File source '{self.source_id}' requires a path."
            )

        if (
            self.access_mode == AccessMode.API
            and not self.url
            and not self.connection_parameters
        ):
            raise ValueError(
                f"API source '{self.source_id}' requires a URL "
                "or connection parameters."
            )

        if self.access_mode in {
            AccessMode.DATABASE,
            AccessMode.COMMAND,
            AccessMode.LIVE_LAB,
        } and not self.connection_parameters:
            raise ValueError(
                f"Source '{self.source_id}' requires "
                "connection_parameters."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)

        result["evidence_type"] = self.evidence_type.value
        result["source_format"] = self.source_format.value
        result["access_mode"] = self.access_mode.value

        return result

##Investigation Policies

In [12]:
@dataclass
class InvestigationPolicy:
    """
    Guardrails and stopping conditions for the investigation workflow.
    """

    remediation_mode: RemediationMode = (
        RemediationMode.RECOMMEND_ONLY
    )

    require_human_approval: bool = True
    minimum_diagnosis_confidence: float = 0.80

    maximum_reasoning_iterations: int = 5
    maximum_hypotheses: int = 5
    maximum_tests_per_hypothesis: int = 5

    require_falsifiable_hypotheses: bool = True
    require_challenger_review: bool = True
    require_evidence_references: bool = True

    def validate(self) -> None:
        if not 0.0 <= self.minimum_diagnosis_confidence <= 1.0:
            raise ValueError(
                "minimum_diagnosis_confidence must be "
                "between 0 and 1."
            )

        if self.maximum_reasoning_iterations < 1:
            raise ValueError(
                "maximum_reasoning_iterations must be at least 1."
            )

        if self.maximum_hypotheses < 2:
            raise ValueError(
                "maximum_hypotheses must be at least 2."
            )

        if self.maximum_tests_per_hypothesis < 1:
            raise ValueError(
                "maximum_tests_per_hypothesis must be at least 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["remediation_mode"] = self.remediation_mode.value
        return result

##Evaluation Configuration

In [13]:
@dataclass
class EvaluationConfig:
    """
    Optional benchmark ground truth and evaluation metrics.

    Production incidents may not have ground truth, so this object
    is optional.
    """

    ground_truth_root_cause: Optional[str] = None
    ground_truth_entity_ids: List[str] = field(
        default_factory=list
    )
    ground_truth_fault_type: Optional[str] = None

    metrics: List[str] = field(
        default_factory=lambda: [
            "top_1_root_cause_accuracy",
            "top_3_root_cause_accuracy",
            "localization_accuracy",
            "evidence_grounding_score",
            "investigation_efficiency",
        ]
    )

    @property
    def has_ground_truth(self) -> bool:
        return self.ground_truth_root_cause is not None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [14]:
@dataclass
class NetworkInvestigationConfig:
    """
    Top-level configuration for one reproducible network
    investigation.

    This is the network-investigation equivalent of CompetitionConfig
    in the Agentic AI Data Science Team.
    """

    investigation_id: str
    investigation_name: str
    environment_name: str
    investigation_type: str

    scenario_provider: str
    scenario_id: str

    incident_seed: IncidentSeed
    evidence_sources: List[EvidenceSourceConfig]

    twin_source_id: Optional[str] = None

    investigation_policy: InvestigationPolicy = field(
        default_factory=InvestigationPolicy
    )

    evaluation_config: Optional[EvaluationConfig] = None

    output_dir: str = "/content/network_investigations"

    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

    @property
    def enabled_sources(self) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.evidence_sources
            if source.enabled
        ]

    @property
    def required_sources(self) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.enabled_sources
            if source.required
        ]

    @property
    def available_evidence_types(self) -> List[str]:
        return sorted(
            {
                source.evidence_type.value
                for source in self.enabled_sources
            }
        )

    @property
    def investigation_output_dir(self) -> Path:
        return Path(self.output_dir) / self.investigation_id

    @property
    def report_path(self) -> Path:
        return (
            self.investigation_output_dir
            / "investigation_report.json"
        )

    @property
    def history_path(self) -> Path:
        return (
            self.investigation_output_dir
            / "investigation_history.json"
        )

    def get_source(
        self,
        source_id: str,
    ) -> EvidenceSourceConfig:
        for source in self.enabled_sources:
            if source.source_id == source_id:
                return source

        raise KeyError(
            f"Evidence source '{source_id}' does not exist "
            "or is disabled."
        )

    def get_sources_by_type(
        self,
        evidence_type: EvidenceType,
    ) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.enabled_sources
            if source.evidence_type == evidence_type
        ]

    def validate(self) -> None:
        if not self.investigation_id.strip():
            raise ValueError(
                "investigation_id cannot be empty."
            )

        if not self.investigation_name.strip():
            raise ValueError(
                "investigation_name cannot be empty."
            )

        if not self.scenario_provider.strip():
            raise ValueError(
                "scenario_provider cannot be empty."
            )

        if not self.scenario_id.strip():
            raise ValueError(
                "scenario_id cannot be empty."
            )

        self.incident_seed.validate()
        self.investigation_policy.validate()

        source_ids = [
            source.source_id
            for source in self.enabled_sources
        ]

        duplicate_source_ids = sorted(
            {
                source_id
                for source_id in source_ids
                if source_ids.count(source_id) > 1
            }
        )

        if duplicate_source_ids:
            raise ValueError(
                "Duplicate evidence source IDs: "
                f"{duplicate_source_ids}"
            )

        for source in self.enabled_sources:
            source.validate()

        if self.twin_source_id is not None:
            twin_source = self.get_source(
                self.twin_source_id
            )

            if twin_source.evidence_type not in {
                EvidenceType.TOPOLOGY,
            }:
                raise ValueError(
                    f"twin_source_id '{self.twin_source_id}' "
                    "must reference a topology source."
                )

    def to_dict(self) -> Dict[str, Any]:
        return {
            "investigation_id": self.investigation_id,
            "investigation_name": self.investigation_name,
            "environment_name": self.environment_name,
            "investigation_type": self.investigation_type,
            "scenario_provider": self.scenario_provider,
            "scenario_id": self.scenario_id,
            "incident_seed": self.incident_seed.to_dict(),
            "evidence_sources": [
                source.to_dict()
                for source in self.evidence_sources
            ],
            "twin_source_id": self.twin_source_id,
            "available_evidence_types": (
                self.available_evidence_types
            ),
            "investigation_policy": (
                self.investigation_policy.to_dict()
            ),
            "evaluation_config": (
                self.evaluation_config.to_dict()
                if self.evaluation_config
                else None
            ),
            "output_dir": self.output_dir,
            "report_path": str(self.report_path),
            "history_path": str(self.history_path),
            "metadata": self.metadata,
        }

    def __repr__(self) -> str:
        return (
            "NetworkInvestigationConfig("
            f"id='{self.investigation_id}', "
            f"provider='{self.scenario_provider}', "
            f"scenario='{self.scenario_id}', "
            f"sources={len(self.enabled_sources)})"
        )

##Synthetic Test

In [15]:
test_investigation_config = NetworkInvestigationConfig(
    investigation_id="synthetic-link-failure-001",
    investigation_name="Synthetic Link Failure",
    environment_name="development_fixture",
    investigation_type="connectivity_failure",

    scenario_provider="synthetic",
    scenario_id="link_failure_001",

    incident_seed=IncidentSeed(
        reported_symptom=(
            "The client host cannot reach the destination service."
        ),
        incident_start_time="2026-08-04T16:00:00Z",
        affected_service="service-app-01",
        reported_locations=["site-sfo"],
        reported_entities=["host-client-01"],
        source="synthetic_fixture",
        initial_severity="high",
    ),

    twin_source_id="synthetic_topology",

    evidence_sources=[
        EvidenceSourceConfig(
            source_id="synthetic_topology",
            evidence_type=EvidenceType.TOPOLOGY,
            source_format=SourceFormat.JSON,
            access_mode=AccessMode.FILE,
            adapter_name="SyntheticEvidenceAdapter",
            path="/content/data/synthetic/topology.json",
            required=True,
        ),
        EvidenceSourceConfig(
            source_id="synthetic_telemetry",
            evidence_type=EvidenceType.TELEMETRY,
            source_format=SourceFormat.CSV,
            access_mode=AccessMode.FILE,
            adapter_name="SyntheticTelemetryAdapter",
            path="/content/data/synthetic/telemetry.csv",
            required=False,
        ),
    ],

    investigation_policy=InvestigationPolicy(
        remediation_mode=RemediationMode.RECOMMEND_ONLY,
        require_human_approval=True,
        minimum_diagnosis_confidence=0.80,
        maximum_reasoning_iterations=5,
        maximum_hypotheses=5,
        maximum_tests_per_hypothesis=5,
    ),

    evaluation_config=EvaluationConfig(
        ground_truth_root_cause="Link failure between R1 and R2",
        ground_truth_entity_ids=["link-r1-r2"],
        ground_truth_fault_type="link_failure",
    ),

    metadata={
        "purpose": "configuration_contract_validation",
    },
)

test_investigation_config.validate()

print(test_investigation_config)
print()
print(
    "Enabled sources:",
    [
        source.source_id
        for source in test_investigation_config.enabled_sources
    ],
)
print(
    "Evidence types:",
    test_investigation_config.available_evidence_types,
)
print(
    "Report path:",
    test_investigation_config.report_path,
)

NetworkInvestigationConfig(id='synthetic-link-failure-001', provider='synthetic', scenario='link_failure_001', sources=2)

Enabled sources: ['synthetic_topology', 'synthetic_telemetry']
Evidence types: ['telemetry', 'topology']
Report path: /content/network_investigations/synthetic-link-failure-001/investigation_report.json


#5. DOMAIN MODEL

In [16]:
@dataclass
class NetworkEntity:
    entity_id: str
    entity_type: EntityType
    name: str

    attributes: Dict[str, Any] = field(default_factory=dict)
    source_ids: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.entity_id.strip():
            raise ValueError("entity_id cannot be empty.")

        if not self.name.strip():
            raise ValueError("name cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["entity_type"] = self.entity_type.value
        return result


@dataclass
class NetworkRelationship:
    relationship_id: str
    source_entity_id: str
    target_entity_id: str
    relationship_type: RelationshipType

    attributes: Dict[str, Any] = field(default_factory=dict)

    valid_from: Optional[str] = None
    valid_to: Optional[str] = None

    source_ids: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.relationship_id.strip():
            raise ValueError("relationship_id cannot be empty.")

        if not self.source_entity_id.strip():
            raise ValueError("source_entity_id cannot be empty.")

        if not self.target_entity_id.strip():
            raise ValueError("target_entity_id cannot be empty.")

        if self.source_entity_id == self.target_entity_id:
            raise ValueError(
                "A relationship cannot connect an entity to itself."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["relationship_type"] = self.relationship_type.value
        return result

In [17]:
@dataclass
class ObservationRecord:
    observation_id: str
    source_id: str
    entity_id: str

    metric_name: str
    metric_value: Any
    observed_at: str

    unit: Optional[str] = None
    dimensions: Dict[str, Any] = field(default_factory=dict)

    quality_score: float = 1.0

    def validate(self) -> None:
        if not self.observation_id.strip():
            raise ValueError("observation_id cannot be empty.")

        if not self.entity_id.strip():
            raise ValueError("entity_id cannot be empty.")

        if not self.metric_name.strip():
            raise ValueError("metric_name cannot be empty.")

        if not 0.0 <= self.quality_score <= 1.0:
            raise ValueError(
                "quality_score must be between 0 and 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class EventRecord:
    event_id: str
    source_id: str
    event_type: str
    occurred_at: str

    entity_ids: List[str] = field(default_factory=list)

    severity: Optional[str] = None
    message: Optional[str] = None
    attributes: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.event_id.strip():
            raise ValueError("event_id cannot be empty.")

        if not self.event_type.strip():
            raise ValueError("event_type cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ChangeRecord:
    change_id: str
    source_id: str
    changed_at: str
    change_type: str

    entity_ids: List[str] = field(default_factory=list)

    previous_state: Any = None
    new_state: Any = None

    actor: Optional[str] = None
    approved: Optional[bool] = None
    rollback_reference: Optional[str] = None

    attributes: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.change_id.strip():
            raise ValueError("change_id cannot be empty.")

        if not self.change_type.strip():
            raise ValueError("change_type cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

##Incident Framing Agent Output

In [18]:
@dataclass
class IncidentFrame:
    """
    Structured output produced by the Incident Framing Agent.

    It converts the raw incident seed into a clear investigation
    problem without attempting to diagnose root cause.
    """

    incident_id: str

    investigation_question: str
    scope_summary: str

    known_facts: List[str] = field(default_factory=list)
    assumptions: List[str] = field(default_factory=list)
    unknowns: List[str] = field(default_factory=list)

    affected_entity_ids: List[str] = field(default_factory=list)
    potentially_relevant_entity_ids: List[str] = field(default_factory=list)

    required_capabilities: List[str] = field(default_factory=list)

    severity: Optional[str] = None

    def validate(self) -> None:
        if not self.incident_id.strip():
            raise ValueError("incident_id cannot be empty.")

        if not self.investigation_question.strip():
            raise ValueError(
                "investigation_question cannot be empty."
            )

        if not self.scope_summary.strip():
            raise ValueError(
                "scope_summary cannot be empty."
            )

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [39]:
def build_incident_framing_context(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: EvidenceGraph,
    topology_capability: Optional[TopologyCapability] = None,
) -> Dict[str, Any]:
    """
    Build the bounded context supplied to the Incident Framing Agent.

    The agent receives normalized evidence and capability output,
    never raw source data.
    """

    seed = investigation_config.incident_seed

    reported_entities = []

    for entity_id in seed.reported_entities:
        try:
            reported_entities.append(
                evidence_graph.get_entity(entity_id).to_dict()
            )

        except KeyError:
            reported_entities.append(
                {
                    "entity_id": entity_id,
                    "status": "not_found_in_evidence_graph",
                }
            )

    topology_evidence = None

    if (
        topology_capability is not None
        and len(seed.reported_entities) >= 2
    ):
        topology_evidence = topology_capability.collect_evidence(
            evidence_graph=evidence_graph,
            source_entity_id=seed.reported_entities[0],
            target_entity_id=seed.reported_entities[1],
        )

    return {
        "investigation_id": (
            investigation_config.investigation_id
        ),
        "investigation_name": (
            investigation_config.investigation_name
        ),
        "investigation_type": (
            investigation_config.investigation_type
        ),

        "reported_symptom": seed.reported_symptom,
        "incident_start_time": seed.incident_start_time,
        "affected_service": seed.affected_service,
        "reported_locations": list(
            seed.reported_locations
        ),
        "reported_entities": reported_entities,
        "initial_severity": seed.initial_severity,

        "available_evidence_types": (
            investigation_config.available_evidence_types
        ),

        "evidence_graph_summary": (
            evidence_graph.summary()
        ),

        "topology_evidence": topology_evidence,

        "available_capabilities": [
            "topology",
        ],
    }

##Hypothesis Domain Object

In [43]:
@dataclass
class HypothesisRecord:
    """
    One falsifiable explanation for the reported incident.
    """

    hypothesis_id: str
    incident_id: str

    title: str
    proposed_cause: str
    causal_mechanism: str

    suspected_entity_ids: List[str] = field(
        default_factory=list
    )

    expected_observations: List[str] = field(
        default_factory=list
    )

    falsifying_observations: List[str] = field(
        default_factory=list
    )

    required_capabilities: List[str] = field(
        default_factory=list
    )

    prior_confidence: float = 0.0

    status: HypothesisStatus = (
        HypothesisStatus.PROPOSED
    )

    def validate(self) -> None:
        if not self.hypothesis_id.strip():
            raise ValueError(
                "hypothesis_id cannot be empty."
            )

        if not self.title.strip():
            raise ValueError(
                "Hypothesis title cannot be empty."
            )

        if not self.proposed_cause.strip():
            raise ValueError(
                "proposed_cause cannot be empty."
            )

        if not self.causal_mechanism.strip():
            raise ValueError(
                "causal_mechanism cannot be empty."
            )

        if not self.falsifying_observations:
            raise ValueError(
                "Every hypothesis must define at least "
                "one falsifying observation."
            )

        if not 0.0 <= self.prior_confidence <= 1.0:
            raise ValueError(
                "prior_confidence must be between 0 and 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["status"] = self.status.value
        return result

In [44]:
@dataclass
class HypothesisSet:
    """
    Competing hypotheses generated for one incident.
    """

    incident_id: str
    hypotheses: List[HypothesisRecord]

    def validate(self) -> None:
        if len(self.hypotheses) < 2:
            raise ValueError(
                "At least two competing hypotheses "
                "are required."
            )

        hypothesis_ids = [
            hypothesis.hypothesis_id
            for hypothesis in self.hypotheses
        ]

        if len(hypothesis_ids) != len(
            set(hypothesis_ids)
        ):
            raise ValueError(
                "Hypothesis IDs must be unique."
            )

        for hypothesis in self.hypotheses:
            hypothesis.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "hypotheses": [
                hypothesis.to_dict()
                for hypothesis in self.hypotheses
            ],
        }

##Hypothesis Generator Context

In [46]:
def build_hypothesis_generation_context(
    incident_frame: IncidentFrame,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> Dict[str, Any]:
    """
    Build bounded evidence context for hypothesis generation.
    """

    topology_evidence = None

    if len(
        incident_frame.affected_entity_ids
    ) >= 2:
        topology_evidence = (
            topology_capability.collect_evidence(
                evidence_graph=evidence_graph,
                source_entity_id=(
                    incident_frame.affected_entity_ids[0]
                ),
                target_entity_id=(
                    incident_frame.affected_entity_ids[1]
                ),
            )
        )

    return {
        "incident_frame": (
            incident_frame.to_dict()
        ),

        "topology_evidence": (
            topology_evidence
        ),

        "available_capabilities": [
            "topology",
        ],
    }

#6. EVIDENCE GRAPH

Agents reason over this graph. Executors populate and query it.

In [20]:
class EvidenceGraph:
    """
    Domain wrapper around a NetworkX MultiDiGraph.

    Agents and executors interact with this domain class rather than
    accessing NetworkX directly.
    """

    def __init__(self) -> None:
        self._graph = nx.MultiDiGraph()

        self._entities: Dict[str, NetworkEntity] = {}
        self._relationships: Dict[str, NetworkRelationship] = {}
        self._observations: Dict[str, ObservationRecord] = {}
        self._events: Dict[str, EventRecord] = {}
        self._changes: Dict[str, ChangeRecord] = {}

    # --------------------------------------------------------
    # Counts and summaries
    # --------------------------------------------------------

    @property
    def entity_count(self) -> int:
        return len(self._entities)

    @property
    def relationship_count(self) -> int:
        return len(self._relationships)

    @property
    def observation_count(self) -> int:
        return len(self._observations)

    @property
    def event_count(self) -> int:
        return len(self._events)

    @property
    def change_count(self) -> int:
        return len(self._changes)

    def summary(self) -> Dict[str, int]:
        """Return basic Evidence Graph record counts."""

        return {
            "entities": self.entity_count,
            "relationships": self.relationship_count,
            "observations": self.observation_count,
            "events": self.event_count,
            "changes": self.change_count,
        }

    # --------------------------------------------------------
    # Entity and relationship mutation
    # --------------------------------------------------------

    def add_entity(self, entity: NetworkEntity) -> None:
        entity.validate()

        if entity.entity_id in self._entities:
            raise ValueError(
                f"Entity '{entity.entity_id}' already exists."
            )

        self._entities[entity.entity_id] = entity

        self._graph.add_node(
            entity.entity_id,
            entity_type=entity.entity_type.value,
            name=entity.name,
            attributes=dict(entity.attributes),
            source_ids=list(entity.source_ids),
        )

    def upsert_entity(self, entity: NetworkEntity) -> None:
        entity.validate()

        self._entities[entity.entity_id] = entity

        self._graph.add_node(
            entity.entity_id,
            entity_type=entity.entity_type.value,
            name=entity.name,
            attributes=dict(entity.attributes),
            source_ids=list(entity.source_ids),
        )

    def add_relationship(
        self,
        relationship: NetworkRelationship,
    ) -> None:
        relationship.validate()

        if relationship.relationship_id in self._relationships:
            raise ValueError(
                f"Relationship "
                f"'{relationship.relationship_id}' already exists."
            )

        if relationship.source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: "
                f"{relationship.source_entity_id}"
            )

        if relationship.target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: "
                f"{relationship.target_entity_id}"
            )

        self._relationships[
            relationship.relationship_id
        ] = relationship

        self._graph.add_edge(
            relationship.source_entity_id,
            relationship.target_entity_id,
            key=relationship.relationship_id,
            relationship_id=relationship.relationship_id,
            relationship_type=relationship.relationship_type.value,
            attributes=dict(relationship.attributes),
            valid_from=relationship.valid_from,
            valid_to=relationship.valid_to,
            source_ids=list(relationship.source_ids),
        )

    # --------------------------------------------------------
    # Evidence mutation
    # --------------------------------------------------------

    def add_observation(
        self,
        observation: ObservationRecord,
    ) -> None:
        observation.validate()

        if observation.observation_id in self._observations:
            raise ValueError(
                f"Observation "
                f"'{observation.observation_id}' already exists."
            )

        if observation.entity_id not in self._entities:
            raise KeyError(
                f"Unknown observation entity: "
                f"{observation.entity_id}"
            )

        self._observations[
            observation.observation_id
        ] = observation

    def add_event(self, event: EventRecord) -> None:
        event.validate()

        if event.event_id in self._events:
            raise ValueError(
                f"Event '{event.event_id}' already exists."
            )

        missing_entities = [
            entity_id
            for entity_id in event.entity_ids
            if entity_id not in self._entities
        ]

        if missing_entities:
            raise KeyError(
                f"Unknown event entities: {missing_entities}"
            )

        self._events[event.event_id] = event

    def add_change(self, change: ChangeRecord) -> None:
        change.validate()

        if change.change_id in self._changes:
            raise ValueError(
                f"Change '{change.change_id}' already exists."
            )

        missing_entities = [
            entity_id
            for entity_id in change.entity_ids
            if entity_id not in self._entities
        ]

        if missing_entities:
            raise KeyError(
                f"Unknown change entities: {missing_entities}"
            )

        self._changes[change.change_id] = change

    # --------------------------------------------------------
    # Entity and relationship queries
    # --------------------------------------------------------

    def get_entity(
        self,
        entity_id: str,
    ) -> NetworkEntity:
        try:
            return self._entities[entity_id]
        except KeyError as exc:
            raise KeyError(
                f"Unknown entity: {entity_id}"
            ) from exc

    def get_relationship(
        self,
        relationship_id: str,
    ) -> NetworkRelationship:
        try:
            return self._relationships[relationship_id]
        except KeyError as exc:
            raise KeyError(
                f"Unknown relationship: {relationship_id}"
            ) from exc

    def list_entities(
        self,
        entity_type: Optional[EntityType] = None,
    ) -> List[NetworkEntity]:
        entities = list(self._entities.values())

        if entity_type is None:
            return entities

        return [
            entity
            for entity in entities
            if entity.entity_type == entity_type
        ]

    def get_neighbors(
        self,
        entity_id: str,
        relationship_type: Optional[
            RelationshipType
        ] = None,
    ) -> List[NetworkEntity]:
        if entity_id not in self._entities:
            raise KeyError(f"Unknown entity: {entity_id}")

        neighbor_ids = set()

        for _, target_id, _, edge_data in self._graph.out_edges(
            entity_id,
            keys=True,
            data=True,
        ):
            if (
                relationship_type is None
                or edge_data.get("relationship_type")
                == relationship_type.value
            ):
                neighbor_ids.add(target_id)

        for source_id, _, _, edge_data in self._graph.in_edges(
            entity_id,
            keys=True,
            data=True,
        ):
            if (
                relationship_type is None
                or edge_data.get("relationship_type")
                == relationship_type.value
            ):
                neighbor_ids.add(source_id)

        return [
            self._entities[neighbor_id]
            for neighbor_id in sorted(neighbor_ids)
        ]

    # --------------------------------------------------------
    # Topology queries
    # --------------------------------------------------------

    def _simple_directed_graph(self) -> nx.DiGraph:
        """
        Return a simple directed projection of the multigraph.

        Parallel relationships are collapsed for path algorithms.
        """

        return nx.DiGraph(self._graph)

    def find_paths(
        self,
        source_entity_id: str,
        target_entity_id: str,
        cutoff: Optional[int] = None,
    ) -> List[List[str]]:
        if source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: {source_entity_id}"
            )

        if target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: {target_entity_id}"
            )

        return list(
            nx.all_simple_paths(
                self._simple_directed_graph(),
                source=source_entity_id,
                target=target_entity_id,
                cutoff=cutoff,
            )
        )

    def shortest_path(
        self,
        source_entity_id: str,
        target_entity_id: str,
    ) -> List[str]:
        if source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: {source_entity_id}"
            )

        if target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: {target_entity_id}"
            )

        try:
            return nx.shortest_path(
                self._simple_directed_graph(),
                source=source_entity_id,
                target=target_entity_id,
            )
        except nx.NetworkXNoPath as exc:
            raise ValueError(
                f"No directed path exists between "
                f"'{source_entity_id}' and "
                f"'{target_entity_id}'."
            ) from exc

    def descendants(
        self,
        entity_id: str,
    ) -> List[str]:
        """Return downstream entities in the directed graph."""

        if entity_id not in self._entities:
            raise KeyError(f"Unknown entity: {entity_id}")

        return sorted(
            nx.descendants(
                self._simple_directed_graph(),
                entity_id,
            )
        )

    # --------------------------------------------------------
    # Evidence queries
    # --------------------------------------------------------

    def get_observations(
        self,
        entity_ids: Optional[List[str]] = None,
        metric_names: Optional[List[str]] = None,
        start_time: Optional[str] = None,
        end_time: Optional[str] = None,
    ) -> List[ObservationRecord]:
        results = list(self._observations.values())

        if entity_ids is not None:
            entity_id_set = set(entity_ids)
            results = [
                record
                for record in results
                if record.entity_id in entity_id_set
            ]

        if metric_names is not None:
            metric_name_set = set(metric_names)
            results = [
                record
                for record in results
                if record.metric_name in metric_name_set
            ]

        if start_time is not None:
            results = [
                record
                for record in results
                if record.observed_at >= start_time
            ]

        if end_time is not None:
            results = [
                record
                for record in results
                if record.observed_at <= end_time
            ]

        return sorted(
            results,
            key=lambda record: record.observed_at,
        )

#7. EVIDENCE ADAPTER LAYER




In [21]:
@dataclass
class EvidenceAdapterResult:
    """
    Standardized output returned by every evidence adapter.

    An adapter may populate some or all record collections depending
    on the source. For example, Topology Zoo may return entities and
    relationships, while Prometheus may primarily return observations.
    """

    source_id: str

    entities: List[NetworkEntity] = field(default_factory=list)
    relationships: List[NetworkRelationship] = field(default_factory=list)
    observations: List[ObservationRecord] = field(default_factory=list)
    events: List[EventRecord] = field(default_factory=list)
    changes: List[ChangeRecord] = field(default_factory=list)

    metadata: Dict[str, Any] = field(default_factory=dict)
    warnings: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.source_id.strip():
            raise ValueError("source_id cannot be empty.")

        for entity in self.entities:
            entity.validate()

        for relationship in self.relationships:
            relationship.validate()

        for observation in self.observations:
            observation.validate()

        for event in self.events:
            event.validate()

        for change in self.changes:
            change.validate()

    def summary(self) -> Dict[str, int]:
        return {
            "entities": len(self.entities),
            "relationships": len(self.relationships),
            "observations": len(self.observations),
            "events": len(self.events),
            "changes": len(self.changes),
            "warnings": len(self.warnings),
        }


class EvidenceAdapter(ABC):
    """
    Abstract translator between a raw evidence source and the
    vendor-neutral Evidence Graph domain model.

    Concrete adapters must:
      1. read a source;
      2. normalize source-specific schemas;
      3. create typed domain records; and
      4. return an EvidenceAdapterResult.

    Adapters do not diagnose incidents or evaluate hypotheses.
    """

    def __init__(
        self,
        source_config: EvidenceSourceConfig,
    ) -> None:
        self.source_config = source_config
        self.source_config.validate()

    @property
    def source_id(self) -> str:
        return self.source_config.source_id

    @abstractmethod
    def load(self) -> EvidenceAdapterResult:
        """
        Load and normalize the configured evidence source.

        Returns
        -------
        EvidenceAdapterResult
            Vendor-neutral entities, relationships, observations,
            events, and changes.
        """
        raise NotImplementedError

    def validate_result(
        self,
        result: EvidenceAdapterResult,
    ) -> None:
        """
        Validate the adapter output and ensure source attribution
        matches the adapter configuration.
        """

        result.validate()

        if result.source_id != self.source_id:
            raise ValueError(
                "Adapter result source_id does not match the "
                f"configured source_id: expected '{self.source_id}', "
                f"received '{result.source_id}'."
            )

In [22]:
def ingest_adapter_result_executor(
    evidence_graph: EvidenceGraph,
    adapter_result: EvidenceAdapterResult,
    *,
    upsert_entities: bool = True,
) -> Dict[str, Any]:
    """
    Insert normalized adapter records into an EvidenceGraph.

    This is an executor because ingestion is deterministic. It does
    not interpret evidence or make root-cause conclusions.
    """

    adapter_result.validate()

    ingested_counts = {
        "entities": 0,
        "relationships": 0,
        "observations": 0,
        "events": 0,
        "changes": 0,
    }

    # Entities must be inserted first because every other record may
    # reference them.
    for entity in adapter_result.entities:
        if upsert_entities:
            evidence_graph.upsert_entity(entity)
        else:
            evidence_graph.add_entity(entity)

        ingested_counts["entities"] += 1

    # Relationships require both endpoint entities to already exist.
    for relationship in adapter_result.relationships:
        evidence_graph.add_relationship(relationship)
        ingested_counts["relationships"] += 1

    for observation in adapter_result.observations:
        evidence_graph.add_observation(observation)
        ingested_counts["observations"] += 1

    for event in adapter_result.events:
        evidence_graph.add_event(event)
        ingested_counts["events"] += 1

    for change in adapter_result.changes:
        evidence_graph.add_change(change)
        ingested_counts["changes"] += 1

    return {
        "executor": "ingest_adapter_result_executor",
        "source_id": adapter_result.source_id,
        "ingested_counts": ingested_counts,
        "warnings": list(adapter_result.warnings),
        "adapter_metadata": dict(adapter_result.metadata),
        "graph_summary": evidence_graph.summary(),
    }

##Adapter Registry

In [23]:
# ============================================================
# ADAPTER REGISTRY
# ============================================================

ADAPTER_REGISTRY: Dict[str, type[EvidenceAdapter]] = {}


def register_adapter(
    adapter_name: str,
    adapter_class: type[EvidenceAdapter],
) -> None:
    """
    Register an EvidenceAdapter implementation by name.
    """

    if not adapter_name.strip():
        raise ValueError("adapter_name cannot be empty.")

    if not issubclass(adapter_class, EvidenceAdapter):
        raise TypeError(
            f"{adapter_class.__name__} must inherit from EvidenceAdapter."
        )

    if adapter_name in ADAPTER_REGISTRY:
        raise ValueError(
            f"Adapter '{adapter_name}' is already registered."
        )

    ADAPTER_REGISTRY[adapter_name] = adapter_class


def get_adapter_class(
    adapter_name: str,
) -> type[EvidenceAdapter]:
    """
    Resolve the adapter class referenced by EvidenceSourceConfig.
    """

    try:
        return ADAPTER_REGISTRY[adapter_name]

    except KeyError as exc:
        raise KeyError(
            f"No adapter registered as '{adapter_name}'. "
            f"Available adapters: {sorted(ADAPTER_REGISTRY)}"
        ) from exc

##Scenario-Specific NIKA Adapter

In [24]:
class NIKASimpleBGPAdapter(EvidenceAdapter):
    """
    Normalize NIKA's simple_bgp scenario into Evidence Graph
    domain records.

    Initial topology:

        pc1 -- router1 -- router2 -- pc2

    This adapter is intentionally narrow. It validates the NIKA
    integration path before we generalize into a reusable NIKAAdapter.
    """

    def load(self) -> EvidenceAdapterResult:
        scenario_id = self.source_config.metadata.get("scenario_id")

        if scenario_id != "simple_bgp":
            raise ValueError(
                "NIKASimpleBGPAdapter only supports "
                "scenario_id='simple_bgp'."
            )

        entities = [
            NetworkEntity(
                entity_id="nika:simple_bgp:pc1",
                entity_type=EntityType.HOST,
                name="pc1",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "host",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:router1",
                entity_type=EntityType.DEVICE,
                name="router1",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "bgp_router",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:router2",
                entity_type=EntityType.DEVICE,
                name="router2",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "bgp_router",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:pc2",
                entity_type=EntityType.HOST,
                name="pc2",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "host",
                },
                source_ids=[self.source_id],
            ),
        ]

        relationships = [
            NetworkRelationship(
                relationship_id="nika:simple_bgp:pc1-router1",
                source_entity_id="nika:simple_bgp:pc1",
                target_entity_id="nika:simple_bgp:router1",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                },
                source_ids=[self.source_id],
            ),

            NetworkRelationship(
                relationship_id="nika:simple_bgp:router1-router2",
                source_entity_id="nika:simple_bgp:router1",
                target_entity_id="nika:simple_bgp:router2",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                    "routing_protocol": "bgp",
                },
                source_ids=[self.source_id],
            ),

            NetworkRelationship(
                relationship_id="nika:simple_bgp:router2-pc2",
                source_entity_id="nika:simple_bgp:router2",
                target_entity_id="nika:simple_bgp:pc2",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                },
                source_ids=[self.source_id],
            ),
        ]

        result = EvidenceAdapterResult(
            source_id=self.source_id,
            entities=entities,
            relationships=relationships,
            metadata={
                "provider": "nika",
                "scenario_id": "simple_bgp",
                "adapter": self.__class__.__name__,
                "integration_stage": "static_topology",
            },
        )

        self.validate_result(result)

        return result

In [25]:
register_adapter(
    "NIKASimpleBGPAdapter",
    NIKASimpleBGPAdapter,
)

print("Registered adapters:")
print(sorted(ADAPTER_REGISTRY))

Registered adapters:
['NIKASimpleBGPAdapter']


#8. DETERMINISTIC EXECUTORS

In [26]:
def entity_lookup_executor(
    evidence_graph: EvidenceGraph,
    entity_id: str,
) -> Dict[str, Any]:
    entity = evidence_graph.get_entity(entity_id)

    return {
        "executor": "entity_lookup_executor",
        "entity": entity.to_dict(),
        "neighbors": [
            neighbor.to_dict()
            for neighbor in evidence_graph.get_neighbors(
                entity_id
            )
        ],
    }


def path_analysis_executor(
    evidence_graph: EvidenceGraph,
    source_entity_id: str,
    target_entity_id: str,
    *,
    include_all_paths: bool = False,
    cutoff: Optional[int] = None,
) -> Dict[str, Any]:
    shortest_path = evidence_graph.shortest_path(
        source_entity_id,
        target_entity_id,
    )

    result = {
        "executor": "path_analysis_executor",
        "source_entity_id": source_entity_id,
        "target_entity_id": target_entity_id,
        "shortest_path": shortest_path,
        "hop_count": len(shortest_path) - 1,
    }

    if include_all_paths:
        all_paths = evidence_graph.find_paths(
            source_entity_id,
            target_entity_id,
            cutoff=cutoff,
        )

        result["all_paths"] = all_paths
        result["path_count"] = len(all_paths)

    return result


def blast_radius_executor(
    evidence_graph: EvidenceGraph,
    failed_entity_id: str,
) -> Dict[str, Any]:
    failed_entity = evidence_graph.get_entity(
        failed_entity_id
    )

    simple_graph = nx.DiGraph(evidence_graph._graph)

    affected_entity_ids = sorted(
        nx.descendants(
            simple_graph,
            failed_entity_id,
        )
    )

    return {
        "executor": "blast_radius_executor",
        "failed_entity": failed_entity.to_dict(),
        "affected_entity_ids": affected_entity_ids,
        "affected_count": len(affected_entity_ids),
    }

In [27]:
test_graph = EvidenceGraph()

test_entities = [
    NetworkEntity(
        entity_id="site-sfo",
        entity_type=EntityType.SITE,
        name="San Francisco Site",
    ),
    NetworkEntity(
        entity_id="host-client-01",
        entity_type=EntityType.HOST,
        name="Client 01",
    ),
    NetworkEntity(
        entity_id="router-r1",
        entity_type=EntityType.DEVICE,
        name="Router R1",
    ),
    NetworkEntity(
        entity_id="router-r2",
        entity_type=EntityType.DEVICE,
        name="Router R2",
    ),
    NetworkEntity(
        entity_id="service-app-01",
        entity_type=EntityType.SERVICE,
        name="Destination Service",
    ),
]

for entity in test_entities:
    test_graph.add_entity(entity)

test_relationships = [
    NetworkRelationship(
        relationship_id="rel-site-client",
        source_entity_id="site-sfo",
        target_entity_id="host-client-01",
        relationship_type=RelationshipType.CONTAINS,
    ),
    NetworkRelationship(
        relationship_id="rel-client-r1",
        source_entity_id="host-client-01",
        target_entity_id="router-r1",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
    NetworkRelationship(
        relationship_id="rel-r1-r2",
        source_entity_id="router-r1",
        target_entity_id="router-r2",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
    NetworkRelationship(
        relationship_id="rel-r2-service",
        source_entity_id="router-r2",
        target_entity_id="service-app-01",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
]

for relationship in test_relationships:
    test_graph.add_relationship(relationship)

test_graph.add_observation(
    ObservationRecord(
        observation_id="obs-r1-utilization",
        source_id="test_telemetry",
        entity_id="router-r1",
        metric_name="cpu_utilization",
        metric_value=42.5,
        unit="percent",
        observed_at="2026-08-04T14:00:00Z",
    )
)

print(test_graph.summary())

print(
    path_analysis_executor(
        evidence_graph=test_graph,
        source_entity_id="host-client-01",
        target_entity_id="service-app-01",
        include_all_paths=True,
    )
)

{'entities': 5, 'relationships': 4, 'observations': 1, 'events': 0, 'changes': 0}
{'executor': 'path_analysis_executor', 'source_entity_id': 'host-client-01', 'target_entity_id': 'service-app-01', 'shortest_path': ['host-client-01', 'router-r1', 'router-r2', 'service-app-01'], 'hop_count': 3, 'all_paths': [['host-client-01', 'router-r1', 'router-r2', 'service-app-01']], 'path_count': 1}


In [28]:
print(
    entity_lookup_executor(
        evidence_graph=test_graph,
        entity_id="router-r1",
    )
)

{'executor': 'entity_lookup_executor', 'entity': {'entity_id': 'router-r1', 'entity_type': 'device', 'name': 'Router R1', 'attributes': {}, 'source_ids': []}, 'neighbors': [{'entity_id': 'host-client-01', 'entity_type': 'host', 'name': 'Client 01', 'attributes': {}, 'source_ids': []}, {'entity_id': 'router-r2', 'entity_type': 'device', 'name': 'Router R2', 'attributes': {}, 'source_ids': []}]}


In [29]:
print(
    blast_radius_executor(
        evidence_graph=test_graph,
        failed_entity_id="router-r1",
    )
)

{'executor': 'blast_radius_executor', 'failed_entity': {'entity_id': 'router-r1', 'entity_type': 'device', 'name': 'Router R1', 'attributes': {}, 'source_ids': []}, 'affected_entity_ids': ['router-r2', 'service-app-01'], 'affected_count': 2}


In [30]:
def load_evidence_source_executor(
    source_config: EvidenceSourceConfig,
    evidence_graph: EvidenceGraph,
) -> Dict[str, Any]:
    """
    Resolve the configured adapter, normalize one source,
    and ingest the result into the Evidence Graph.
    """

    source_config.validate()

    adapter_class = get_adapter_class(
        source_config.adapter_name
    )

    adapter = adapter_class(
        source_config=source_config
    )

    adapter_result = adapter.load()

    adapter.validate_result(
        adapter_result
    )

    ingestion_result = ingest_adapter_result_executor(
        evidence_graph=evidence_graph,
        adapter_result=adapter_result,
    )

    return {
        "executor": "load_evidence_source_executor",
        "source_id": source_config.source_id,
        "adapter_name": source_config.adapter_name,
        "adapter_summary": adapter_result.summary(),
        "ingestion_result": ingestion_result,
    }

In [31]:
def load_investigation_evidence_executor(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: Optional[EvidenceGraph] = None,
) -> Dict[str, Any]:
    """
    Load every enabled evidence source configured for an investigation.

    Required-source failures stop execution.
    Optional-source failures are recorded and skipped.
    """

    investigation_config.validate()

    if evidence_graph is None:
        evidence_graph = EvidenceGraph()

    source_results: Dict[str, Any] = {}
    source_errors: Dict[str, str] = {}

    for source_config in investigation_config.enabled_sources:

        try:
            result = load_evidence_source_executor(
                source_config=source_config,
                evidence_graph=evidence_graph,
            )

            source_results[
                source_config.source_id
            ] = result

            print(
                f"✓ Loaded {source_config.source_id} "
                f"using {source_config.adapter_name}"
            )

        except Exception as exc:
            source_errors[
                source_config.source_id
            ] = str(exc)

            if source_config.required:
                raise RuntimeError(
                    "Required evidence source failed: "
                    f"{source_config.source_id}"
                ) from exc

            print(
                f"⚠ Optional evidence source unavailable: "
                f"{source_config.source_id}"
            )

    return {
        "executor": "load_investigation_evidence_executor",
        "investigation_id": (
            investigation_config.investigation_id
        ),
        "source_results": source_results,
        "source_errors": source_errors,
        "graph_summary": evidence_graph.summary(),
        "evidence_graph": evidence_graph,
    }

In [32]:
nika_simple_bgp_config = NetworkInvestigationConfig(
    investigation_id="nika-simple-bgp-link-down-001",
    investigation_name="NIKA Simple BGP Link Failure",
    environment_name="nika",
    investigation_type="connectivity_failure",

    scenario_provider="nika",
    scenario_id="simple_bgp",

    incident_seed=IncidentSeed(
        reported_symptom=(
            "Users report connectivity issues between "
            "the two endpoint hosts."
        ),
        incident_start_time="2026-08-07T00:00:00Z",
        reported_entities=[
            "nika:simple_bgp:pc1",
            "nika:simple_bgp:pc2",
        ],
        source="nika",
        initial_severity="high",
    ),

    twin_source_id="nika_simple_bgp_topology",

    evidence_sources=[
        EvidenceSourceConfig(
            source_id="nika_simple_bgp_topology",
            evidence_type=EvidenceType.TOPOLOGY,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="NIKASimpleBGPAdapter",

            connection_parameters={
                "provider": "nika",
            },

            required=True,

            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
            },
        ),
    ],

    investigation_policy=InvestigationPolicy(
        remediation_mode=RemediationMode.RECOMMEND_ONLY,
        require_human_approval=True,
        minimum_diagnosis_confidence=0.80,
    ),

    evaluation_config=EvaluationConfig(
        ground_truth_root_cause="link_down",
        ground_truth_fault_type="link_failure",
    ),

    metadata={
        "nika_task": "simple_bgp_link_down",
        "integration_stage": "static_topology",
    },
)

In [33]:
nika_simple_bgp_config.validate()

nika_load_result = load_investigation_evidence_executor(
    investigation_config=nika_simple_bgp_config
)

nika_graph = nika_load_result["evidence_graph"]

print()
print("Graph summary:")
print(nika_graph.summary())

✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter

Graph summary:
{'entities': 4, 'relationships': 3, 'observations': 0, 'events': 0, 'changes': 0}


In [34]:
nika_path_result = path_analysis_executor(
    evidence_graph=nika_graph,
    source_entity_id="nika:simple_bgp:pc1",
    target_entity_id="nika:simple_bgp:pc2",
    include_all_paths=True,
)

print(nika_path_result)

{'executor': 'path_analysis_executor', 'source_entity_id': 'nika:simple_bgp:pc1', 'target_entity_id': 'nika:simple_bgp:pc2', 'shortest_path': ['nika:simple_bgp:pc1', 'nika:simple_bgp:router1', 'nika:simple_bgp:router2', 'nika:simple_bgp:pc2'], 'hop_count': 3, 'all_paths': [['nika:simple_bgp:pc1', 'nika:simple_bgp:router1', 'nika:simple_bgp:router2', 'nika:simple_bgp:pc2']], 'path_count': 1}


#9. INVESTIGATION CAPABILITIES

Capabilities provide deterministic investigative services to AI agents.

Agents request information needs (e.g. topology, telemetry, events)
rather than invoking individual executors directly.

Capabilities may internally orchestrate one or more deterministic
executors and return a single normalized result.

In [35]:
from typing import Optional


class TopologyCapability:
    """
    Deterministic topology investigation capability.

    Provides a simplified interface over topology-related executors.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        source_entity_id: str,
        target_entity_id: Optional[str] = None,
    ) -> Dict[str, Any]:

        result = {
            "capability": "topology",
        }

        # --------------------------------------------------
        # Source entity
        # --------------------------------------------------

        result["source_entity"] = entity_lookup_executor(
            evidence_graph=evidence_graph,
            entity_id=source_entity_id,
        )

        # --------------------------------------------------
        # Blast radius
        # --------------------------------------------------

        result["blast_radius"] = blast_radius_executor(
            evidence_graph=evidence_graph,
            failed_entity_id=source_entity_id,
        )

        # --------------------------------------------------
        # Optional path analysis
        # --------------------------------------------------

        if target_entity_id is not None:

            result["path_analysis"] = (
                path_analysis_executor(
                    evidence_graph=evidence_graph,
                    source_entity_id=source_entity_id,
                    target_entity_id=target_entity_id,
                )
            )

        return result

testing the topology capability

In [36]:
topology_capability = TopologyCapability()

topology_result = topology_capability.collect_evidence(
    evidence_graph=nika_graph,
    source_entity_id="nika:simple_bgp:router1",
    target_entity_id="nika:simple_bgp:pc2",
)

print(topology_result)

{'capability': 'topology', 'source_entity': {'executor': 'entity_lookup_executor', 'entity': {'entity_id': 'nika:simple_bgp:router1', 'entity_type': 'device', 'name': 'router1', 'attributes': {'provider': 'nika', 'scenario': 'simple_bgp', 'role': 'bgp_router'}, 'source_ids': ['nika_simple_bgp_topology']}, 'neighbors': [{'entity_id': 'nika:simple_bgp:pc1', 'entity_type': 'host', 'name': 'pc1', 'attributes': {'provider': 'nika', 'scenario': 'simple_bgp', 'role': 'host'}, 'source_ids': ['nika_simple_bgp_topology']}, {'entity_id': 'nika:simple_bgp:router2', 'entity_type': 'device', 'name': 'router2', 'attributes': {'provider': 'nika', 'scenario': 'simple_bgp', 'role': 'bgp_router'}, 'source_ids': ['nika_simple_bgp_topology']}]}, 'blast_radius': {'executor': 'blast_radius_executor', 'failed_entity': {'entity_id': 'nika:simple_bgp:router1', 'entity_type': 'device', 'name': 'router1', 'attributes': {'provider': 'nika', 'scenario': 'simple_bgp', 'role': 'bgp_router'}, 'source_ids': ['nika_simp

#10. AI AGENTS

##Incident Framing Agent

In [40]:
def incident_framing_agent(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> IncidentFrame:
    """
    Frame the incident into a structured investigation problem.

    This agent must not diagnose root cause.
    """

    context = build_incident_framing_context(
        investigation_config=investigation_config,
        evidence_graph=evidence_graph,
        topology_capability=topology_capability,
    )

    system_prompt = """
You are the Incident Framing Agent for an autonomous network
investigation team.

Your job is to transform the reported network symptom into a precise,
bounded investigation problem.

You must distinguish:

- reported facts;
- assumptions;
- unknowns;
- affected entities;
- potentially relevant entities; and
- investigative capabilities that may be needed.

Do NOT diagnose root cause.
Do NOT recommend remediation.
Do NOT generate detailed hypotheses.

Return only a Python dictionary with exactly these keys:

{
    "investigation_question": str,
    "scope_summary": str,
    "known_facts": list[str],
    "assumptions": list[str],
    "unknowns": list[str],
    "affected_entity_ids": list[str],
    "potentially_relevant_entity_ids": list[str],
    "required_capabilities": list[str],
    "severity": str | None
}

Use only entity IDs that appear in the supplied context.
Only request capabilities listed under available_capabilities.
"""

    user_prompt = f"""
Investigation context:

{context}

Frame this network incident.
"""

    response = llm_call(
        agent_name="incident_framing_agent",
        messages=[{"role": "system",
              "content": system_prompt,},
            {"role": "user",
              "content": user_prompt,},
        ],temperature=0.1,
    )

    content = response["content"]


    if not content:
        raise ValueError(
            "Incident Framing Agent returned empty LLM content."
        )

    # --------------------------------------------------------
    # 5. Parse structured output
    # --------------------------------------------------------

    cleaned = clean_llm_dict_output(content)
    parsed = ast.literal_eval(cleaned)

    # --------------------------------------------------------
    # 6. Construct typed domain object
    # --------------------------------------------------------

    incident_frame = IncidentFrame(
        incident_id=(f"{investigation_config.investigation_id}-incident-frame"),
        investigation_question=parsed["investigation_question"],
        scope_summary=parsed[
            "scope_summary"
        ],
        known_facts=parsed.get(
            "known_facts",
            [],
        ),
        assumptions=parsed.get(
            "assumptions",
            [],
        ),
        unknowns=parsed.get(
            "unknowns",
            [],
        ),

        affected_entity_ids=parsed.get(
            "affected_entity_ids",
            [],
        ),

        potentially_relevant_entity_ids=parsed.get(
            "potentially_relevant_entity_ids",
            [],
        ),

        required_capabilities=parsed.get(
            "required_capabilities",
            [],
        ),

        severity=parsed.get(
            "severity"
        ),
    )

    # --------------------------------------------------------
    # 7. Validate
    # --------------------------------------------------------

    incident_frame.validate()

    # --------------------------------------------------------
    # 8. RETURN THE RESULT
    # --------------------------------------------------------

    return incident_frame



testing the incident framing agent

In [42]:
incident_frame = incident_framing_agent(
    investigation_config=nika_simple_bgp_config,
    evidence_graph=nika_graph,
    topology_capability=topology_capability,
)

pprint.pprint(incident_frame)

llm call completed
IncidentFrame(incident_id='nika-simple-bgp-link-down-001-incident-frame',
              investigation_question='Why is connectivity between pc1 and pc2 '
                                     'failing, and where in the path (pc1 → '
                                     'router1 → router2 → pc2) is the failure '
                                     'occurring?',
              scope_summary='Investigation of reported connectivity failure '
                            'between two endpoint hosts (pc1 and pc2) in a '
                            'simple BGP topology. The known path traverses two '
                            'BGP routers. Scope is limited to the four '
                            'entities in the topology and their direct '
                            'relationships.',
              known_facts=['Users report connectivity issues between pc1 and '
                           'pc2',
                           'Incident started at 2026-08-07T00:00:00Z',
      

##Hypothesis Generator Agent

In [49]:
@observe(name="hypothesis_generator_agent")
def hypothesis_generator_agent(
    incident_frame: IncidentFrame,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> HypothesisSet:
    """
    Generate competing, falsifiable explanations for the incident.

    This agent proposes hypotheses only.
    It does not decide which hypothesis is correct.
    """

    agent_name = "hypothesis_generator_agent"

    context = build_hypothesis_generation_context(
        incident_frame=incident_frame,
        evidence_graph=evidence_graph,
        topology_capability=topology_capability,
    )

    system_prompt = """
You are the Hypothesis Generator Agent for an autonomous
network investigation team.

Your responsibility is to generate multiple competing,
falsifiable explanations for a framed network incident.

Important rules:

1. Generate between 3 and 5 competing hypotheses.
2. Do NOT select a root cause.
3. Do NOT recommend remediation.
4. Each hypothesis must explain a plausible causal mechanism.
5. Each hypothesis must specify observable evidence that would
   support it.
6. Each hypothesis must specify at least one observation that
   would falsify it.
7. Use only entity IDs present in the supplied context.
8. Request only capabilities listed under available_capabilities.
9. Do not treat benchmark ground truth as investigation evidence.
10. Avoid duplicate hypotheses that describe the same failure
    using different wording.

Return ONLY a Python dictionary with exactly this shape:

{
    "hypotheses": [
        {
            "title": str,
            "proposed_cause": str,
            "causal_mechanism": str,
            "suspected_entity_ids": list[str],
            "expected_observations": list[str],
            "falsifying_observations": list[str],
            "required_capabilities": list[str],
            "prior_confidence": float
        }
    ]
}

prior_confidence must be between 0.0 and 1.0.

The confidence values represent initial plausibility only.
They do not need to sum to 1.0.
"""

    user_prompt = f"""
Investigation context:

{context}

Generate competing hypotheses for this incident.
"""

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {"role": "system",
                "content": system_prompt,},
            {"role": "user",
                "content": user_prompt,},],
        temperature=0.2,)

    print("Hypothesis generation LLM call completed")

    content = response["content"]

    if not content:
        raise ValueError(
            "Incident Framing Agent returned empty LLM content."
        )


    cleaned = clean_llm_dict_output(content)
    parsed = ast.literal_eval(cleaned)

    raw_hypotheses = parsed.get(
        "hypotheses",
        []
    )

    hypotheses = []

    for index, item in enumerate(
        raw_hypotheses,
        start=1,
    ):
        hypothesis = HypothesisRecord(
            hypothesis_id=(
                f"{incident_frame.incident_id}-h{index}"
            ),

            incident_id=incident_frame.incident_id,

            title=item["title"],

            proposed_cause=item[
                "proposed_cause"
            ],

            causal_mechanism=item[
                "causal_mechanism"
            ],

            suspected_entity_ids=item.get(
                "suspected_entity_ids",
                [],
            ),

            expected_observations=item.get(
                "expected_observations",
                [],
            ),

            falsifying_observations=item.get(
                "falsifying_observations",
                [],
            ),

            required_capabilities=item.get(
                "required_capabilities",
                [],
            ),

            prior_confidence=float(
                item.get(
                    "prior_confidence",
                    0.0,
                )
            ),
        )

        hypothesis.validate()
        hypotheses.append(hypothesis)

    hypothesis_set = HypothesisSet(
        incident_id=incident_frame.incident_id,
        hypotheses=hypotheses,
    )

    hypothesis_set.validate()

    return hypothesis_set

testing the hypothesis generator agent

In [51]:
hypothesis_set = hypothesis_generator_agent(
    incident_frame=incident_frame,
    evidence_graph=nika_graph,
    topology_capability=topology_capability,
)

pprint.pprint(hypothesis_set)

llm call completed
Hypothesis generation LLM call completed
HypothesisSet(incident_id='nika-simple-bgp-link-down-001-incident-frame',
              hypotheses=[HypothesisRecord(hypothesis_id='nika-simple-bgp-link-down-001-incident-frame-h1',
                                           incident_id='nika-simple-bgp-link-down-001-incident-frame',
                                           title='BGP Route Withdrawal or '
                                                 'Non-Advertisement',
                                           proposed_cause='router1 has '
                                                          'withdrawn or is not '
                                                          'advertising the '
                                                          "route to pc2's "
                                                          'network to router2, '
                                                          'or router2 has '
                                             

#11. INVESTIGATION WORKFLOW

#12. VALIDATION